# 02 — Activations and Metadata

**Purpose:** Walk through every field on an `Op` record that carries payload, shape,
dtype, device, memory, timing, compute, args, RNG state, and source-code context.
Shows `Quantity` formatting (`Bytes`, `Duration`, `Flops`, `Macs`) via f-strings.

**Surfaces covered:**
- [ ] `op.out` — captured output tensor
- [ ] `op.shape` — output shape tuple
- [ ] `op.dtype` — torch dtype
- [ ] `op.device_ref` — `DeviceRef` (`.backend`, `.name`)
- [ ] `op.activation_memory` — raw byte count
- [ ] `Bytes` / `Duration` / `Flops` / `Macs` — `Quantity` classes + f-string formatting
- [ ] `op.func_duration` — wall-clock time for the op
- [ ] `op.flops_forward` / `op.macs_forward` — compute counts
- [ ] `op.arg_expressions` — symbolic arg names
- [ ] `op.saved_args` — captured input tensors (requires `save_arg_values=True`)
- [ ] `op.arg_names` — parameter-style names (e.g. `('input', 'weight', 'bias')`)
- [ ] `op.func_rng_states` — RNG state dict (requires `save_rng_states=True`)
- [ ] `op.code_context` — list of `FuncCallLocation` records
- [ ] `FuncCallLocation` — `.call_line`, `.code_context`, repr

## 1. Environment setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from torchlens.types import Bytes, Duration, Flops, Macs
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. Capture a trace — baseline model

Use `tiny_mlp` for simple fields; `demo_model` later for RNG state.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

print(repr(trace))
print("layer_labels:", trace.layer_labels)

# Grab the linear and relu ops for metadata inspection
op_lin = trace["linear_1_1"]  # linear op
op_relu = trace["relu_1_2"]  # relu op

## 3. `op.out` — output tensor

The captured output tensor.  Returned as a detached copy by default so it is safe to
inspect and mutate independently of the computation graph.

In [ ]:
print("op_relu.out type :", type(op_relu.out).__name__)
print("op_relu.out      :")
print(op_relu.out)
print()
print("Is grad retained in .out?", op_relu.out.grad_fn)

## 4. `op.shape`, `op.dtype`, `op.device_ref`

In [ ]:
print("op_lin.shape      :", op_lin.shape)
print("op_lin.dtype      :", op_lin.dtype)
print()
dev = op_lin.device_ref
print("op_lin.device_ref :", dev)
print("  type            :", type(dev).__name__)
print("  .backend        :", dev.backend)
print("  .name           :", dev.name)

## 5. `op.activation_memory` and `Bytes` formatting

`activation_memory` is the raw byte count of the output tensor.  
`Bytes` is a `Quantity` subclass with a human-readable `__str__` / `__format__`.

In [ ]:
raw_bytes = op_lin.activation_memory
print("op_lin.activation_memory (raw int):", raw_bytes)

b = Bytes(raw_bytes)
print("Bytes(raw)        :", b)  # auto format
print(f"f-string          : {b}")  # same via __format__
print()

# Show scaling at different sizes
for n_bytes in [512, 2**10, 2**20, 2**30]:
    print(f"  {n_bytes:>12,} B  ->  {Bytes(n_bytes)}")

## 6. `Duration` — wall-clock timing per op

`op.func_duration` holds seconds as a float; wrapping in `Duration` humanises it.

In [ ]:
raw_dur = op_lin.func_duration
print("op_lin.func_duration (raw float, seconds):", raw_dur)

d = Duration(raw_dur)
print("Duration(raw)  :", d)
print(f"f-string       : {d}")
print()

# Duration across every op in the trace
print("Per-op timing:")
for lbl in trace.layer_labels:
    op = trace[lbl]
    print(f"  {lbl:<18}: {Duration(op.func_duration)}")

## 7. `Flops` and `Macs` — compute counts

`op.flops_forward` counts FLOPs (convention: one multiply-accumulate = 2 FLOPs).  
`op.macs_forward` is the MAC count (= FLOPs // 2).  
Both are 0 for non-compute ops (input, output).

In [ ]:
raw_flops = op_lin.flops_forward
raw_macs = op_lin.macs_forward
print("op_lin.flops_forward (raw):", raw_flops)
print("op_lin.macs_forward  (raw):", raw_macs)
print()

fl = Flops(raw_flops)
mc = Macs(raw_macs)
print("Flops(raw) :", fl)
print(f"f-string   : {fl}")
print("Macs(raw)  :", mc)
print(f"f-string   : {mc}")
print()

# Compute summary across all ops
print("Per-op compute:")
for lbl in trace.layer_labels:
    op = trace[lbl]
    print(f"  {lbl:<18}: FLOPs={Flops(op.flops_forward)}  MACs={Macs(op.macs_forward)}")

## 8. `op.arg_names` and `op.arg_expressions`

`arg_names` are the formal parameter names from the torch function signature  
(e.g. `('input', 'weight', 'bias')` for `F.linear`).  
`arg_expressions` are symbolic expressions describing what was passed  
(e.g. `['input', 'self.weight', 'self.bias']`).  

They are populated during default tracing.  `saved_args` (the actual tensors) requires
`save_arg_values=True` (see section 9).

In [ ]:
print("=== linear op ===")
print("  arg_names       :", op_lin.arg_names)
print("  arg_expressions :", op_lin.arg_expressions)
print("  has_saved_args  :", op_lin.has_saved_args)
print()

print("=== relu op ===")
print("  arg_names       :", op_relu.arg_names)
print("  arg_expressions :", op_relu.arg_expressions)

## 9. `op.saved_args` — captured input tensors (`save_arg_values=True`)

By default `saved_args` is `None` to avoid storing every intermediate tensor twice.
Pass `save_arg_values=True` to `tl.trace` to retain them.

In [ ]:
model2, x2 = ZOO["tiny_mlp"]()
trace_args = tl.trace(model2, x2, save_arg_values=True)

op_lin2 = trace_args["linear_1_1"]
print("op_lin2.has_saved_args :", op_lin2.has_saved_args)
print("op_lin2.saved_args     :", op_lin2.saved_args)
print()

if op_lin2.saved_args is not None:
    for i, t in enumerate(op_lin2.saved_args):
        print(f"  arg[{i}] shape={t.shape} dtype={t.dtype}")

## 10. `op.func_rng_states` — RNG state at call time (`save_rng_states=True`)

Captured only for ops that consume randomness (e.g. `torch.rand`, `dropout`).  
Useful for exact replay.  Requires `save_rng_states=True`.

In [ ]:
# demo_model has torch.rand in its forward pass
model_d, x_d = ZOO["demo_model"]()
trace_rng = tl.trace(model_d, x_d, save_rng_states=True)

rand_labels = [lbl for lbl in trace_rng.layer_labels if "rand" in lbl]
print("rand-type labels:", rand_labels)

if rand_labels:
    op_rand = trace_rng[rand_labels[0]]
    print()
    print("op_rand.func_name      :", op_rand.func_name)
    rng = op_rand.func_rng_states
    print("op_rand.func_rng_states type :", type(rng).__name__)
    print("  keys  :", list(rng.keys()))
    for k, v in rng.items():
        if hasattr(v, "shape"):
            print(f"  {k}: tensor shape={v.shape} dtype={v.dtype}")
        else:
            print(f"  {k}: {type(v).__name__} len={len(v) if hasattr(v, '__len__') else '?'}")
else:
    print("No rand op found in demo_model trace -- check model forward()")

## 11. `op.code_context` and `FuncCallLocation`

`code_context` is a list of `FuncCallLocation` records tracing the Python call stack
at the point where the op was invoked.  By default only the innermost frame is captured.
Pass `save_code_context=True` for the full stack.  Each `FuncCallLocation` has:
- `.call_line` — 1-indexed line number in source
- `.code_context` — the source line text (or `None` if unavailable for dynamic code)
- repr shows file:line:function

In [ ]:
import torch.nn as nn


# Use a named module so source is available
class ExplicitModel(nn.Module):
    """Tiny model defined in-notebook so code_context has a real source line."""

    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(self.fc(x))  # <-- this line captured in code_context


model_e = ExplicitModel()
x_e = torch.randn(2, 4)
trace_ctx = tl.trace(model_e, x_e, save_code_context=True)

op_ctx = trace_ctx["relu"]
cc = op_ctx.code_context

print(f"code_context: {len(cc)} frame(s)")
for i, fl in enumerate(cc):
    print(f"\n--- frame {i} ---")
    print("  repr        :", repr(fl))
    print("  .call_line  :", fl.call_line)
    print("  .code_context:", fl.code_context)

## 12. Full metadata snapshot on one op

Collect all the above fields into a single printout so JMT can see what a user
encounters when they call `print(vars(op))` or iterate fields.

In [ ]:
model3, x3 = ZOO["tiny_mlp"]()
trace_full = tl.trace(model3, x3, save_arg_values=True, save_code_context=True)
op_full = trace_full["linear_1_1"]

print("=== Full metadata snapshot: linear_1_1 ===")
print(f"  func_name          : {op_full.func_name}")
print(f"  layer_label        : {op_full.layer_label}")
print(f"  raw_label          : {op_full.raw_label}")
print(f"  shape              : {op_full.shape}")
print(f"  dtype              : {op_full.dtype}")
print(f"  device_ref.name    : {op_full.device_ref.name}")
print(f"  activation_memory  : {Bytes(op_full.activation_memory)}")
print(f"  func_duration      : {Duration(op_full.func_duration)}")
print(f"  flops_forward      : {Flops(op_full.flops_forward)}")
print(f"  macs_forward       : {Macs(op_full.macs_forward)}")
print(f"  arg_names          : {op_full.arg_names}")
print(f"  arg_expressions    : {op_full.arg_expressions}")
print(f"  has_saved_args     : {op_full.has_saved_args}")
saved = op_full.saved_args
if saved:
    print(f"  saved_args shapes  : {[tuple(t.shape) for t in saved]}")
cc = op_full.code_context
if cc:
    print(f"  code_context[0]    : {repr(cc[0])}")

---

## ⚠️ GAPs / ergonomic smells

- **`Bytes.__repr__` returns the raw int** (`Bytes(32)` shows `32`, not `'32 B'`).
  `str()` and f-strings work perfectly, but `repr()` returns the bare number.  A user
  inspecting a `Bytes` in a REPL will see just `32` with no unit hint.  `__repr__`
  should probably return `Bytes(32 B)` or at least `'32 B'`.

- **`op.arg_expressions` is an empty list for most ops** in a default trace (requires
  either `save_arg_values=True` or some ops do populate it — e.g. `linear` shows
  `['input', 'self.weight', 'self.bias']` in some traces but not others).  The
  behavior seems inconsistent: in some runs the field populates without a flag, in
  others it is empty.  Worth clarifying the exact condition.

- **`FuncCallLocation.code_context` is `None` for dynamically-defined models**
  (lambda/`<string>` source).  The `repr` shows `code: source unavailable` which is
  clear, but `.code_context` returning `None` (vs `"source unavailable"`) makes it
  awkward to format — callers must check for `None` before using it.

- **`op.func_rng_states` is `{}` by default** even for RNG ops unless `save_rng_states=True`.
  That is correct behavior, but there is no hint in `op.__repr__` that RNG states exist
  and could be captured.  A note like `rng_states: not captured (pass save_rng_states=True)`
  in the repr would help discoverability.